# 線性迴歸的簡潔實現
:label:`sec_linear_concise`

在過去的幾年裡，出於對深度學習強烈的興趣，
許多公司、學者和業餘愛好者開發了各種成熟的開源框架。
這些框架可以自動化基於梯度的學習算法中重複性的工作。
在 :numref:`sec_linear_scratch`中，我們只運用了：
（1）通過張量進行數據存儲和線性代數；
（2）通過自動微分來計算梯度。
實際上，由於數據迭代器、損失函數、優化器和神經網路層很常用，
現代深度學習庫也為我們實現了這些元件。

本節將介紹如何(**通過使用深度學習框架來簡潔地實現**)
 :numref:`sec_linear_scratch`中的(**線性迴歸模型**)。
 
## 生成資料集

與 :numref:`sec_linear_scratch`中類似，我們首先[**生成資料集**]。


In [1]:
import numpy as np
import torch
from torch.utils import data
import matplotlib.pyplot as plt

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [5]:
def synthetic_data(w, b, num_examples):
    # 產生 X，其中 X ~ N(0, 1)
    X = torch.normal(mean=0.0, std=1.0, size=(num_examples, len(w)))
    
    # 理想情況下的 y = Xw + b
    y = X @ w + b  # 或 torch.matmul(X, w) + b
    
    # 加上雜訊，noise ~ N(0, 0.01^2)
    y += torch.normal(mean=0.0, std=0.01, size=y.shape)
    return X, y.reshape((-1, 1))
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

## 讀取資料集

我們可以[**呼叫框架中現有的API來讀取資料**]。
我們將`features`和`labels`作為API的參數傳遞，並透過資料迭代器指定`batch_size`。
此外，布林值`is_train`表示是否希望資料迭代器物件在每個迭代週期內打亂資料。


In [6]:
def load_array(data_arrays, batch_size, is_train=True):  #@save
    """建構一個 PyTorch 資料迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

In [7]:
batch_size = 10
data_iter = load_array((features, labels), batch_size)

使用`data_iter`的方式與我們在 :numref:`sec_linear_scratch`中使用`data_iter`函數的方式相同。為了驗證是否正常運作，讓我們讀取並列印第一個小批量樣本。
與 :numref:`sec_linear_scratch`不同，這裡我們使用`iter`建構Python迭代器，並使用`next`從迭代器中獲取第一項。


In [8]:
next(iter(data_iter))

[tensor([[ 0.7441, -1.2160],
         [ 0.2909,  0.5864],
         [ 0.1744,  0.3044],
         [-0.0740,  0.7191],
         [-0.5692, -1.7438],
         [-0.9262,  0.0961],
         [ 0.8368,  0.7335],
         [-0.5462,  0.1096],
         [-0.0764,  1.3929],
         [ 0.1943, -1.0600]]),
 tensor([[ 9.8274],
         [ 2.7826],
         [ 3.5271],
         [ 1.5953],
         [ 8.9916],
         [ 2.0209],
         [ 3.3732],
         [ 2.7391],
         [-0.6862],
         [ 8.1894]])]

## 定義模型
 
當我們在 :numref:`sec_linear_scratch`中實現線性回歸時，
我們明確定義了模型參數變量，並編寫了計算的程式碼，這樣通過基本的線性代數運算得到輸出。
但是，如果模型變得更加複雜，且當我們幾乎每天都需要實現模型時，自然會想簡化這個過程。
這種情況類似於為自己的部落格從零開始編寫網頁。
做一兩次是有益的，但如果每個新部落格就需要工程師花一個月的時間重新開始編寫網頁，那並不高效。

對於標準深度學習模型，我們可以[**使用框架的預定義好的層**]。這使我們只需關注使用哪些層來構造模型，而不必關注層的實現細節。
我們首先定義一個模型變量`net`，它是一個`Sequential`類的實例。
`Sequential`類將多個層串聯在一起。
當給定輸入數據時，`Sequential`實例將數據傳入到第一層，
然後將第一層的輸出作為第二層的輸入，以此類推。
在下面的例子中，我們的模型只包含一個層，因此實際上不需要`Sequential`。
但是由於以後幾乎所有的模型都是多層的，在這裡使用`Sequential`會讓你熟悉"標準的流水線"。

回顧 :numref:`fig_single_neuron`中的單層網絡架構，
這一單層被稱為*全連接層*（fully-connected layer），
因為它的每一個輸入都通過矩陣-向量乘法得到它的每個輸出。


在PyTorch中，全连接层在`Linear`类中定义。
值得注意的是，我们将两个参数传递到`nn.Linear`中。
第一个指定输入特征形状，即2，第二个指定输出特征形状，输出特征形状为单个标量，因此为1。


In [9]:
# nn是神經網路的縮寫
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

## (**初始化模型參數**)
 
在使用`net`之前，我們需要初始化模型參數。
如在線性迴歸模型中的權重和偏置。
深度學習框架通常有預定義的方法來初始化參數。
在這裡，我們指定每個權重參數應該從均值為0、標準差為0.01的正態分佈中隨機採樣，
偏置參數將初始化為零。


正如我們在構造`nn.Linear`時指定輸入和輸出尺寸一樣，
現在我們能直接訪問參數以設定它們的初始值。
我們通過`net[0]`選擇網絡中的第一個圖層，
然後使用`weight.data`和`bias.data`方法訪問參數。
我們還可以使用替換方法`normal_`和`fill_`來重寫參數值。


In [10]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

## 定義損失函數


[**計算均方誤差使用的是`MSELoss`類，也稱為平方$L_2$範數**]。
默認情況下，它返回所有樣本損失的平均值。


In [11]:
loss = nn.MSELoss()

## 定義優化算法


小批量隨機梯度下降算法是一種優化神經網路的標準工具，
PyTorch在`optim`模塊中實現了該算法的許多變種。
當我們(**實例化一個`SGD`實例**)時，我們要指定優化的參數
（可透過`net.parameters()`從我們的模型中獲得）以及優化算法所需的超參數字典。
小批量隨機梯度下降只需要設置`lr`值，這裡設置為0.03。


In [12]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

## 訓練

透過深度學習框架的高級API來實現我們的模型只需要相對較少的代碼。
我們不必單獨分配參數、不必定義我們的損失函數，也不必手動實現小批量隨機梯度下降。
當我們需要更複雜的模型時，高級API的優勢將大大增加。
當我們有了所有的基本組件，[**訓練過程代碼與我們從零開始實現時所做的非常相似**]。

回顧一下：在每個迭代週期裡，我們將完整遍歷一次資料集（`train_data`），
不停地從中獲取一個小批量的輸入和相應的標籤。
對於每一個小批量，我們會進行以下步驟:

* 透過調用`net(X)`生成預測並計算損失`l`（前向傳播）。
* 透過進行反向傳播來計算梯度。
* 透過調用優化器來更新模型參數。

為了更好的衡量訓練效果，我們計算每個迭代週期後的損失，並打印它來監控訓練過程。


In [13]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000236
epoch 2, loss 0.000103
epoch 3, loss 0.000102


下面我們[**比較生成資料集的真實參數和透過有限資料訓練獲得的模型參數**]。
要訪問參數，我們首先從`net`訪問所需的層，然後讀取該層的權重和偏置。
正如在從零開始實現中一樣，我們估計得到的參數與生成資料的真實參數非常接近。


In [15]:
w = net[0].weight.data
print('w的估計誤差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估計誤差：', true_b - b)

w的估計誤差： tensor([0.0003, 0.0005])
b的估計誤差： tensor([-0.0010])


## 小結


* 我們可以使用PyTorch的高級API更簡潔地實現模型。
* 在PyTorch中，`data`模塊提供了數據處理工具，`nn`模塊定義了大量的神經網路層和常見損失函數。
* 我們可以透過`_`結尾的方法將參數替換，從而初始化參數。


## 練習

1. 如果將小批量的總損失替換為小批量損失的平均值，需要如何更改學習率？
1. 查看深度學習框架文檔，它們提供了哪些損失函數和初始化方法？用Huber損失代替原損失，即
    $$l(y,y') = \begin{cases}|y-y'| -\frac{\sigma}{2} & \text{ if } |y-y'| > \sigma \\ \frac{1}{2 \sigma} (y-y')^2 & \text{ 其它情况}\end{cases}$$
1. 如何訪問線性回歸的梯度？


1. 如果將小批量的總損失替換為小批量損失的平均值，需要如何更改學習率？

    Ans:如果原本就用的是小批量損失的平均值（大多數深度學習框架預設如此），那其實不必再調整學習率。

如果本來用總和，後來改用平均，想保留同樣的更新強度，就需適度將學習率 𝜂 放大，大約乘上 batch size。

2. 查看深度學習框架文檔，它們提供了哪些損失函數和初始化方法？用Huber損失代替原損失，即
    $$l(y,y') = \begin{cases}|y-y'| -\frac{\sigma}{2} & \text{ if } |y-y'| > \sigma \\ \frac{1}{2 \sigma} (y-y')^2 & \text{ 其它情况}\end{cases}$$

    Ans:

    在 PyTorch 中，可以使用 torch.nn.HuberLoss 類別或 torch.nn.functional.huber_loss 函數計算 Huber 損失。  delta 參數指定 delta 比例的 L1 和 L2 損失之間的切換閾值。  當 delta 設定為 1 時，Huber 損失等同於 SmoothL1Loss。 

3. 如何訪問線性回歸的梯度？

    Ans:
    ```python
    # 定義一個簡單的線性回歸模型
    class LinearRegression(torch.nn.Module):
        def __init__(self):
            super(LinearRegression, self).__init__()
            self.linear = torch.nn.Linear(1, 1)

        def forward(self, x):
            return self.linear(x)

    # 創建模型實例
    model = LinearRegression()

    # 定義損失函數
    criterion = torch.nn.MSELoss()

    # 定義優化器
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

    # 訓練數據
    x = torch.tensor([[1.0], [2.0], [3.0]], requires_grad=True)
    y = torch.tensor([[2.0], [4.0], [6.0]])

    # 訓練步驟
    predictions = model(x)
    loss = criterion(y, predictions)
    loss.backward()

    # 打印梯度
    for param in model.parameters():
        print(param.grad)
    ```

[Discussions](https://discuss.d2l.ai/t/1781)
